# External Asset Readback

Use this notebook to check that GEECS external asset Resource/Datum documents can be filled locally into image arrays. Server-side Tiled handler installation is intentionally out of scope here.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import png

from geecs_bluesky.assets import (
    GEECS_CAMERA_IMAGE,
    build_camera_shot_documents,
    fill_geecs_documents,
)

## Fill Documents From A Run

`DOCUMENTS` should be a Bluesky document stream in `(name, doc)` order. This can come from a RunEngine callback during a notebook scan, a local document cache, or a Tiled/databroker client once that retrieval path is chosen.

In [ ]:
DOCUMENTS = []
ROOT_MAP = {}
IMAGE_FIELD = ""

In [ ]:
filled_documents = fill_geecs_documents(
    DOCUMENTS,
    root_map=ROOT_MAP,
    retry_intervals=[0.001, 0.01, 0.1, 0.5, 1.0],
)
events = [doc for name, doc in filled_documents if name == "event"]
available_fields = sorted(events[0]["data"]) if events else []

if not events:
    print("No documents configured yet. Set DOCUMENTS to a run document stream.")
else:
    if not IMAGE_FIELD:
        image_candidates = [
            field for field in available_fields if field.endswith("-image")
        ]
        IMAGE_FIELD = image_candidates[0] if image_candidates else available_fields[0]
    image = events[0]["data"][IMAGE_FIELD]
    print(
        IMAGE_FIELD,
        type(image),
        getattr(image, "shape", None),
        getattr(image, "dtype", None),
    )

In [ ]:
if events and IMAGE_FIELD:
    plt.imshow(events[0]["data"][IMAGE_FIELD], cmap="gray")
    plt.axis("off")

## Existing Scan Camera Shot

Set these parameters to an existing camera shot. The helper resolves the scan folder through `geecs_data_utils.ScanPaths`, queries the live GEECS DB for the device type, builds Resource/Datum/Event docs, and fills the image through the local handler. It supports both legacy `ScanNNN_Device_001.png` files and Bluesky/native `Device_<acq_timestamp>.png` files.

In [ ]:
RUN_ARCHIVED_CAMERA_LOOKUP = False

EXPERIMENT = "Undulator"
YEAR = 2026
MONTH = 6
DAY = 23
SCAN_NUMBER = 1
DEVICE_NAME = "UC_Amp2_IR_input"
SHOT_NUMBER = 1
ACQ_TIMESTAMP = None
BASE_DIRECTORY = None

In [ ]:
camera_image = None
camera_field = ""
camera_file = None

if RUN_ARCHIVED_CAMERA_LOOKUP:
    camera_docs, camera_file = build_camera_shot_documents(
        year=YEAR,
        month=MONTH,
        day=DAY,
        scan_number=SCAN_NUMBER,
        device_name=DEVICE_NAME,
        shot_number=SHOT_NUMBER,
        acq_timestamp=ACQ_TIMESTAMP,
        experiment=EXPERIMENT,
        base_directory=BASE_DIRECTORY,
    )
    filled_camera_docs = fill_geecs_documents(camera_docs, retry_intervals=[])
    camera_event = next(doc for name, doc in filled_camera_docs if name == "event")
    camera_field = next(iter(camera_event["data"]))
    camera_image = camera_event["data"][camera_field]

    print(camera_file)
    print(camera_field, type(camera_image), camera_image.shape, camera_image.dtype)
else:
    print("Set RUN_ARCHIVED_CAMERA_LOOKUP = True after entering a real scan.")

In [ ]:
if camera_image is not None:
    plt.imshow(camera_image, cmap="gray")
    plt.axis("off")

## Local Synthetic Smoke Test

This creates a temporary GEECS-style camera PNG and a minimal Resource/Datum/Event stream, then fills the datum ID into a NumPy array. It does not touch hardware or Tiled.

In [ ]:
with TemporaryDirectory() as temp_dir:
    root = Path(temp_dir) / "Scan001"
    image_path = root / "UC_TopView" / "UC_TopView_1.000.png"
    image_path.parent.mkdir(parents=True)

    expected = np.array([[1, 2], [3, 4]], dtype=np.uint8)
    with image_path.open("wb") as stream:
        png.Writer(width=2, height=2, greyscale=True, bitdepth=8).write(
            stream, expected.tolist()
        )

    start_uid = str(uuid4())
    descriptor_uid = str(uuid4())
    resource_uid = str(uuid4())
    datum_id = f"{resource_uid}/0"

    docs = [
        ("start", {"uid": start_uid, "time": 0}),
        (
            "descriptor",
            {
                "uid": descriptor_uid,
                "run_start": start_uid,
                "time": 0,
                "name": "primary",
                "data_keys": {
                    "uc_topview-image": {
                        "source": "geecs://UC_TopView/image",
                        "dtype": "array",
                        "shape": [],
                        "external": "OLD:",
                    }
                },
                "configuration": {},
                "object_keys": {"uc_topview": ["uc_topview-image"]},
                "hints": {},
            },
        ),
        (
            "resource",
            {
                "uid": resource_uid,
                "spec": GEECS_CAMERA_IMAGE,
                "root": str(root),
                "resource_path": "UC_TopView/UC_TopView_1.000.png",
                "resource_kwargs": {"data_key": "uc_topview-image"},
                "path_semantics": "posix",
            },
        ),
        ("datum", {"datum_id": datum_id, "resource": resource_uid, "datum_kwargs": {}}),
        (
            "event",
            {
                "uid": str(uuid4()),
                "descriptor": descriptor_uid,
                "time": 0,
                "seq_num": 1,
                "data": {"uc_topview-image": datum_id},
                "timestamps": {"uc_topview-image": 0},
                "filled": {"uc_topview-image": False},
            },
        ),
        (
            "stop",
            {
                "uid": str(uuid4()),
                "run_start": start_uid,
                "time": 1,
                "exit_status": "success",
                "num_events": {"primary": 1},
            },
        ),
    ]

    filled = fill_geecs_documents(docs, retry_intervals=[])
    event = next(doc for name, doc in filled if name == "event")
    image = event["data"]["uc_topview-image"]
    np.testing.assert_array_equal(image, expected)
    print(type(image), image.shape, image.dtype)